# Notebook 5 — Evaluation

**Project:** IntelliSys Ltd. — London Road Collision Severity Prediction  
**Module:** WM9B7-15 Artificial Intelligence & Deep Learning  
**University:** WMG, University of Warwick — MSc Applied AI 2025/26

---

## Objective

Evaluate the best `CollisionMLP` on the **held-out test set** — data that was  
never seen during training, validation, or hyperparameter selection. Compare against  
the Logistic Regression baseline from Notebook 3.

Every metric is interpreted in the context of road safety and IntelliSys's  
deployment scenario — not reported as an abstract number.

**Primary metric: Fatal recall.**  
A false negative on Fatal (predicting 'Slight' when the crash is actually fatal)  
means no ambulance pre-alert, no dynamic speed limit reduction, no warning signage  
activation. This is the most dangerous model error in this use case.

---
## Step 1 — Imports and Seeds

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score,
    accuracy_score, f1_score, RocCurveDisplay
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import set_seeds, outputs_dir
from src.model import CollisionMLP
from src.evaluate import (
    predict, print_classification_report, plot_confusion_matrix,
    compute_auc_roc, build_comparison_table, plot_training_curves
)
from src.preprocessing import CLASS_NAMES

np.random.seed(42)
torch.manual_seed(42)
set_seeds(42)

PROCESSED_DIR = project_root / 'data' / 'processed'
FIGURES_DIR   = outputs_dir('figures')
MODELS_DIR    = outputs_dir('models')
print('Setup complete.')

---
## Step 2 — Load Test Data and Saved Predictions

In [ ]:
X_train = np.load(PROCESSED_DIR / 'X_train.npy')
X_test  = np.load(PROCESSED_DIR / 'X_test.npy')
y_test  = np.load(PROCESSED_DIR / 'y_test.npy')

# Baseline predictions from Notebook 3
y_pred_lr = np.load(PROCESSED_DIR / 'y_pred_lr.npy')
probs_lr  = np.load(PROCESSED_DIR / 'probs_lr.npy')

with open(PROCESSED_DIR / 'feature_names.txt') as f:
    feature_names = f.read().splitlines()

print(f'Test set: {X_test.shape[0]:,} samples | {X_test.shape[1]} features')
print(f'Test class distribution:')
vals, cnts = np.unique(y_test, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'  {CLASS_NAMES[v]:8s}: {c:4d} ({100*c/len(y_test):.1f}%)')

---
## Step 3 — Load Best MLP and Run Inference

The best model weights were saved to `outputs/models/best_model.pt` by Notebook 4.  
We reconstruct the architecture from the saved config and load the weights.

In [ ]:
# Load best config
best_config = pd.read_csv(PROCESSED_DIR / 'best_config.csv', index_col=0, header=None).squeeze()
print('Best model configuration:')
print(best_config.to_string())

INPUT_DIM = X_test.shape[1]

# Reconstruct model and load weights
best_model = CollisionMLP(
    input_dim=INPUT_DIM,
    hidden1=int(best_config.get('hidden1', 128)),
    hidden2=int(best_config.get('hidden2', 64)),
    dropout=float(best_config.get('dropout', 0.3))
)
best_model.load_state_dict(
    torch.load(MODELS_DIR / 'best_model.pt', map_location='cpu')
)
best_model.eval()
print(f'\nModel loaded. Parameters: {best_model.count_parameters():,}')

In [ ]:
# Run inference on test set
y_pred_mlp, probs_mlp = predict(best_model, X_test)

print(f'MLP predictions generated for {len(y_pred_mlp):,} test samples.')
print(f'Predicted class distribution:')
vals, cnts = np.unique(y_pred_mlp, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'  {CLASS_NAMES[v]:8s}: {c:4d} ({100*c/len(y_pred_mlp):.1f}%)')

---
## Step 4 — Classification Report

**What to look for:**
- **Fatal precision/recall** — the minority class most critical for deployment safety.
- **Serious recall** — second-priority; missed serious injuries also have consequences.
- **Macro F1** — unweighted average across classes; penalises poor minority-class performance.

In [ ]:
print('=== MLP — TEST SET CLASSIFICATION REPORT ===')
print_classification_report(y_test, y_pred_mlp)

print('\n=== LOGISTIC REGRESSION — TEST SET CLASSIFICATION REPORT ===')
print_classification_report(y_test, y_pred_lr)

---
## Step 5 — Confusion Matrices

The confusion matrix shows the full distribution of prediction errors.  
For road safety deployment, the most dangerous cells are:
- **Fatal predicted as Slight** (row 0, col 2): ambulance not pre-alerted, no speed reduction.
- **Serious predicted as Slight** (row 1, col 2): significant under-response to serious injury risk.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title in [
    (axes[0], y_pred_lr,  'Logistic Regression Baseline'),
    (axes[1], y_pred_mlp, 'CollisionMLP (Best Configuration)')
]:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(title, fontweight='bold')

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_comparison.png', dpi=150)
plt.show()

# Save MLP confusion matrix separately
plot_confusion_matrix(y_test, y_pred_mlp)

**Interpretation:** The top-left cell (Fatal correctly predicted as Fatal) directly  
represents collisions where IntelliSys's system would correctly trigger preventive  
action. The cell at row=Fatal, col=Slight represents fatal collisions where no alert  
would fire — the most critical failure mode.

---
## Step 6 — ROC Curves (One-vs-Rest)

ROC curves are plotted one-vs-rest for each class. For the Fatal class,  
the area under the curve (AUC) quantifies the model's ability to rank fatal  
collisions higher than non-fatal ones, independent of the classification threshold.  
An AUC of 0.5 = random; 1.0 = perfect.

In [ ]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
colors = ['#d62728', '#ff7f0e', '#2ca02c']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, probs, label in [(axes[0], probs_lr, 'Logistic Regression'), (axes[1], probs_mlp, 'CollisionMLP')]:
    for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
        RocCurveDisplay.from_predictions(
            y_test_bin[:, i], probs[:, i],
            name=cls, color=color, ax=ax
        )
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')
    ax.set_title(f'ROC Curves (OvR) — {label}', fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')

plt.suptitle('One-vs-Rest ROC Curves — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_curves.png', dpi=150)
plt.show()

auc_mlp = compute_auc_roc(y_test, probs_mlp)
auc_lr  = roc_auc_score(y_test, probs_lr, multi_class='ovr', average='macro')
print(f'Macro AUC-ROC — LR:  {auc_lr:.4f}')
print(f'Macro AUC-ROC — MLP: {auc_mlp:.4f}')

---
## Step 7 — MLP vs Logistic Regression Comparison Table

This table is the key deliverable of Notebook 5. It must appear in the presentation  
slides (Slides 9–11) and justifies the additional complexity of the deep learning approach.

In [ ]:
comparison_table = build_comparison_table(
    y_test=y_test,
    y_pred_lr=y_pred_lr,   y_pred_mlp=y_pred_mlp,
    probs_lr=probs_lr,     probs_mlp=probs_mlp
)

comparison_table['Improvement'] = (
    comparison_table['MLP'] - comparison_table['Logistic Regression']
).round(4)

print('\nModel Comparison — Test Set:')
display(comparison_table.style
    .format('{:.4f}')
    .applymap(lambda v: 'color: green; font-weight: bold' if v > 0 else 'color: red',
              subset=['Improvement'])
    .set_caption('CollisionMLP vs Logistic Regression Baseline — All metrics on held-out test set')
)

comparison_table.to_csv(PROCESSED_DIR / 'model_comparison.csv')
print('Comparison table saved.')

---
## Step 8 — Clinical and Operational Interpretation

### What does each metric mean for IntelliSys?

**Fatal Recall (most important):**  
Fatal recall measures the fraction of actual fatal collisions that the model correctly  
predicts as Fatal. A value of, say, 0.65 means the model would trigger preventive  
alerts for 65% of fatal-risk conditions — leaving 35% undetected. In IntelliSys's  
system, an undetected fatal-risk corridor means ambulance stations are not pre-alerted,  
dynamic speed limits are not lowered, and warning signage is not activated. The  
improvement in Fatal recall from Logistic Regression to MLP directly quantifies the  
operational safety benefit of the deep learning approach.

**Serious Recall:**  
Serious injury collisions represent ~14% of London crashes. Missing these means  
failing to pre-position paramedic resources and not adjusting signal timings on  
high-risk corridors. While less critical than Fatal misses, systematic under-detection  
of Serious collisions would have measurable impact on emergency response times.

**Macro F1:**  
Macro F1 averages F1 equally across all three classes, regardless of their frequency.  
It is the appropriate headline metric for this imbalanced problem — it penalises the  
model equally for poor performance on Fatal (1% of data) as for poor performance  
on Slight (85% of data). This is why we do not report accuracy as the primary metric.

**Accuracy caveat:**  
A model that always predicts 'Slight' would achieve ~85% accuracy on this dataset  
while being completely useless for IntelliSys. Accuracy is included for completeness  
but should not be used to assess deployment readiness.

**Macro AUC-ROC:**  
AUC-ROC measures ranking quality independently of any classification threshold.  
For IntelliSys's confidence-gated alert system (Notebook 6 recommendation),  
a high AUC-ROC means the model's softmax probabilities are well-calibrated —  
genuine high-risk predictions have high probability scores, enabling the 0.6  
confidence threshold to be a meaningful operational filter.

### Class Imbalance and Weighted Loss Effect

The weighted CrossEntropyLoss forces the MLP to treat each fatal collision  
as approximately 40–50× more important than a slight collision during training  
(reflecting the inverse class frequencies). This intentionally trades some  
Slight precision for improved Fatal and Serious recall — the correct trade-off  
for a safety-critical deployment. The confusion matrices should show this:  
the MLP makes more Slight → Fatal/Serious errors (false alarms) but fewer  
Fatal/Serious → Slight errors (missed detections) compared to the baseline.

**False alarm interpretation:** An over-alert (predicting Fatal when the collision  
is actually Slight) results in unnecessary ambulance pre-positioning or temporary  
speed limit reductions — operationally costly but not harmful. A missed fatal  
prediction results in delayed emergency response — potentially life-threatening.  
The asymmetric cost structure justifies erring on the side of over-alerting.

In [ ]:
# Probability calibration check — distribution of max softmax confidence
max_probs_mlp = probs_mlp.max(axis=1)
max_probs_lr  = probs_lr.max(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, probs, label in [(axes[0], max_probs_lr, 'Logistic Regression'), (axes[1], max_probs_mlp, 'CollisionMLP')]:
    ax.hist(probs, bins=40, color='#4c72b0', edgecolor='white', linewidth=0.5)
    ax.axvline(0.6, color='red', linestyle='--', linewidth=1.5, label='Confidence threshold (0.6)')
    pct_above = (probs >= 0.6).mean() * 100
    ax.set_xlabel('Max class probability')
    ax.set_ylabel('Count')
    ax.set_title(f'{label}\n({pct_above:.1f}% predictions above threshold)', fontweight='bold')
    ax.legend()
plt.suptitle('Prediction Confidence Distribution — Test Set', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confidence_distribution.png', dpi=150)
plt.show()

print(f'MLP: {(max_probs_mlp >= 0.6).mean()*100:.1f}% of predictions above 0.6 confidence threshold')
print(f'LR:  {(max_probs_lr  >= 0.6).mean()*100:.1f}% of predictions above 0.6 confidence threshold')

**Interpretation:** Predictions above the 0.6 confidence threshold are those that  
would trigger automated preventive actions in IntelliSys's deployment. Predictions  
below are flagged for human traffic controller review. A well-calibrated model  
should concentrate low-confidence predictions on genuinely ambiguous cases —  
typically Serious/Slight boundary cases — while being highly confident on Fatal predictions.

**Proceed to Notebook 6 — Interpretability and Ethics** for SHAP, LIME, and the full  
ethical assessment required for Distinction/Outstanding marks.